In [7]:
import torch
import torch.nn as nn
import torch.optim as optim

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, input_shape, num_classes):
        super(SimpleCNN, self).__init__()
        # 6 convolutional layers with filters [16, 32, 64, 128, 256, 512]
        # Kernel size 3, pooling size 2
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        self.conv4 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.pool4 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        self.conv5 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.pool5 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        self.conv6 = nn.Conv2d(256, 512, kernel_size=3, padding=1)
        self.pool6 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc1 = nn.Linear(512, 256)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(256, num_classes)
    
    def forward(self, x):
        # Remove any trailing singleton dimensions
        x = x.squeeze()
        # Add channel dimension if needed
        if x.dim() == 3:
            x = x.unsqueeze(1)
        
        # 6 convolutional blocks
        x = torch.relu(self.conv1(x))
        x = self.pool1(x)
        
        x = torch.relu(self.conv2(x))
        x = self.pool2(x)
        
        x = torch.relu(self.conv3(x))
        x = self.pool3(x)
        
        x = torch.relu(self.conv4(x))
        x = self.pool4(x)
        
        x = torch.relu(self.conv5(x))
        x = self.pool5(x)
        
        x = torch.relu(self.conv6(x))
        x = self.pool6(x)
        
        # Global average pooling and fully connected layers
        x = self.global_pool(x)
        x = x.view(x.size(0), -1)  # Flatten
        x = torch.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

In [14]:
optim
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SimpleCNN(input_shape=(128, 128), num_classes=10)
model = model.to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

In [15]:
# Create dummy data and train
from tqdm import tqdm

batch_size = 16
num_batches = 5
num_epochs = 3

# Generate dummy training data
dummy_X = torch.randn(batch_size * num_batches, 1, 128, 128)  # (batch_size, channels, height, width)
dummy_y = torch.randint(0, 10, (batch_size * num_batches,))   # 10 classes

from torch.utils.data import TensorDataset, DataLoader
train_dataset = TensorDataset(dummy_X, dummy_y)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

train_losses = []

for epoch in tqdm(range(num_epochs), desc="Epochs", unit="epoch"):
    model.train()
    epoch_loss = 0.0
    
    batch_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}", leave=False, unit="batch")
    for batch_X, batch_y in batch_pbar:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        batch_pbar.set_postfix({"loss": f"{loss.item():.4f}"})
    
    avg_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_loss)
    print(f"Epoch {epoch+1}/{num_epochs}, Avg Loss: {avg_loss:.4f}")

print("\n✅ Training completed!")
print(f"Final loss: {train_losses[-1]:.4f}")

Epochs:   0%|          | 0/3 [00:00<?, ?epoch/s]

Epochs:   0%|          | 0/3 [00:09<?, ?epoch/s]


KeyboardInterrupt: 